# Transformer-WAF — End-to-End Experiment Pipeline (Colab)

Runs the whole study on a free GPU and prints **paste-ready result tables** for
the paper. Self-contained: nothing to clone.

**Before running:** Runtime -> Change runtime type -> **GPU (T4)**.

Order: Setup -> Get dataset -> Preprocess -> Train -> Evaluate -> Baselines ->
Threshold -> Latency -> Adversarial -> **Results summary**.


## 1. Setup

In [ ]:
!pip -q install transformers datasets scikit-learn pandas pyarrow accelerate >/dev/null 2>&1
import torch, transformers, sklearn, pandas as pd, numpy as np
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime->GPU")
SEED=42
import random; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Get the CSIC 2010 dataset

Three ways - run the one that works. It needs the two raw files
`normalTrafficTraining.txt` and `anomalousTrafficTest.txt`, **or** a single CSV
with a URL/payload column and a label column.

**Option A (recommended): upload manually.** Download CSIC 2010 from Kaggle
(search *"CSIC 2010 HTTP dataset"*) or any mirror, then run the upload cell.

**Option B: Kaggle API** if you have a `kaggle.json` token.

**Option C: auto-download** attempts a public mirror.

In [ ]:
# ---- Option A: manual upload (run, then choose your file(s)) ----
from google.colab import files
import os
os.makedirs('data/raw', exist_ok=True)
print("Select CSIC file(s): the two .txt files, OR a single .csv")
up = files.upload()
for name in up:
    os.rename(name, f'data/raw/{name}')
    print("saved -> data/raw/"+name)
print("Files now in data/raw:", os.listdir('data/raw'))

In [ ]:
# ---- Option B: Kaggle API (optional; skip if you used Option A) ----
# from google.colab import files; files.upload()   # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip -q install kaggle
# !kaggle datasets download -d ispangler/csic-2010-web-application-attacks -p data/raw --unzip
# import os; print(os.listdir('data/raw'))

In [ ]:
# ---- Option C: try a public mirror (skip if Option A worked) ----
import os, urllib.request
os.makedirs('data/raw', exist_ok=True)
CANDIDATES = {
  'normalTrafficTraining.txt': [
    'https://raw.githubusercontent.com/lucasmsp/machine-learning/master/DecisionTree/HTTP-dataset-CSIC-2010/normalTrafficTraining.txt',
  ],
  'anomalousTrafficTest.txt': [
    'https://raw.githubusercontent.com/lucasmsp/machine-learning/master/DecisionTree/HTTP-dataset-CSIC-2010/anomalousTrafficTest.txt',
  ],
}
for fname, urls in CANDIDATES.items():
    if os.path.exists(f'data/raw/{fname}'):
        print("already have", fname); continue
    for u in urls:
        try:
            urllib.request.urlretrieve(u, f'data/raw/{fname}')
            print("downloaded", fname); break
        except Exception as e:
            print("failed", u, "->", e)
print("data/raw:", os.listdir('data/raw'))

## 3. Rule engine + preprocessing (mirrors the deployed WAF)

In [ ]:
import re
from urllib.parse import unquote_plus

def preprocess_request(method, uri):
    uri = (uri or "")[:4096]
    return f"{(method or 'GET').upper()} {re.sub(r'[0-9]+','{ID}',uri)}"

def double_decode(u): return unquote_plus(unquote_plus(u or ""))

SUSPICIOUS=["union","select","drop","insert","delete","update","exec","script",
 "eval","alert","onclick","onerror","onload","passwd","shadow","or 1=1","and 1=1",
 "../","..\\","%00","--",";","|"]
DIRECT=["or 1=1","and 1=1","union select","<script","javascript:","../","..\\","%00"]
INJ=[r"('\s*(AND|OR)\s*'?[^']*'?\s*=|\bOR\b.*=|AND\s*\d+=\d+)",
 r"(UNION\s+SELECT|UNION\s+ALL|UNION\s*/\*)",
 r"(DROP|DELETE|INSERT|UPDATE|EXEC|SCRIPT)\s+",
 r"(' ;\s*(DROP|DELETE|TRUNCATE|INSERT))",
 r"([`;&|]|\|\||&&|\n|\r)\s*(cat|ls|wget|curl|bash|sh|cmd)",
 r"(<\s*script[^>]*>|javascript:|onerror\s*=|onclick\s*=|onload\s*=|eval\()"]
ENC=[r"\\x[0-9a-f]{2}",r"%2e%2e|%252e%252e",r"%2f|%5c|%252f|%255c",r"%00|%2500",
 r"%3c|%3e|%22|%27|%3b|%2d%2d",r"\.\./|\.\\\\",r"&#[0-9]+;"]

def signals(u):
    d=double_decode(u); dl=d.lower()
    kw=any(k in dl for k in SUSPICIOUS)
    enc=any(re.search(p,u or "",re.I) for p in ENC)
    inj=any(re.search(p,d,re.I) for p in INJ)
    sc=sum(1 for c in d if c in "!@#$%^&*()[]{};<>|\\\"'-,./?=")
    ca=sc>10
    return dict(kw=kw,enc=enc,inj=inj,ca=ca,has=kw or enc or inj or ca)

def rules_verdict(method,uri):
    d=double_decode(uri); n=d.lower()
    if any(s in n for s in DIRECT): return 1
    s=signals(uri)
    if s['kw'] and s['inj']: return 1
    if s['enc']: return 1
    if s['ca'] and s['kw']: return 1
    return 0

def fuse(method,uri,p,th=0.98,tl=0.90):
    d=double_decode(uri); n=d.lower()
    if any(x in n for x in DIRECT): return 1
    s=signals(uri)
    if (s['kw'] and s['inj']) or s['enc']: return 1
    if p>th and s['has']: return 1
    if tl<=p<th and (s['kw'] or s['ca']): return 1
    if s['ca'] and s['kw']: return 1
    return 0
print("rule engine ready")

## 4. Parse dataset -> unified schema -> 70/15/15 splits

In [ ]:
import glob, os
from sklearn.model_selection import train_test_split

def attack_type(u):
    u=(u or "").lower()
    if any(k in u for k in ["union","select","or 1=1","and 1=1","drop","'--","sleep("]):return "sqli"
    if any(k in u for k in ["<script","onerror","onload","alert(","javascript:","<img","<svg"]):return "xss"
    if any(k in u for k in ["../","..\\","etc/passwd","etc/shadow","%2e%2e"]):return "path_traversal"
    if any(k in u for k in ["%00","%3c","%3e","&#","%27","%22"]):return "encoding"
    return "other"

REQ=re.compile(r'^(GET|POST|PUT|DELETE|HEAD|PATCH)\s+(\S+)\s+HTTP',re.I)
def parse_raw(path,label):
    rows=[];m=u=None;bm=False;body=""
    for line in open(path,encoding='latin-1'):
        g=REQ.match(line.strip())
        if g:
            if m is not None: rows.append((m,u+("?"+body if body else ""),label))
            m,u=g.group(1).upper(),g.group(2);bm=False;body=""
        elif m=="POST" and line.strip()=="": bm=True
        elif bm and line.strip(): body=line.strip()
    if m is not None: rows.append((m,u+("?"+body if body else ""),label))
    out=[]
    for me,ur,la in rows:
        ur=re.sub(r'^https?://[^/]+','',ur) or "/"
        out.append(dict(method=me,raw_uri=ur,label=la))
    return pd.DataFrame(out)

raw=set(os.listdir('data/raw')) if os.path.isdir('data/raw') else set()
if {'normalTrafficTraining.txt','anomalousTrafficTest.txt'} <= raw:
    df=pd.concat([parse_raw('data/raw/normalTrafficTraining.txt',0),
                  parse_raw('data/raw/anomalousTrafficTest.txt',1)],ignore_index=True)
else:
    csvs=glob.glob('data/raw/*.csv')
    assert csvs, "No raw .txt pair and no .csv in data/raw. Re-run Section 2."
    d=pd.read_csv(csvs[0]); print("CSV columns:",list(d.columns))
    URL_COL=input("URL/payload column name: ").strip()
    LAB_COL=input("label column name: ").strip()
    df=pd.DataFrame({'method':'GET','raw_uri':d[URL_COL].astype(str)})
    lab=d[LAB_COL].astype(str).str.lower().str.strip()
    df['label']=lab.isin(['1','1.0','anom','anomalous','attack','malicious','true']).astype(int)

df['method']=df['method'].fillna('GET').str.upper()
df['text']=[preprocess_request(m,u) for m,u in zip(df.method,df.raw_uri)]
df['attack_type']=[('benign' if l==0 else attack_type(u)) for l,u in zip(df.label,df.raw_uri)]
df=df.drop_duplicates('text').reset_index(drop=True)

tr,tmp=train_test_split(df,train_size=.70,stratify=df.label,random_state=SEED)
va,te=train_test_split(tmp,train_size=.5,stratify=tmp.label,random_state=SEED)
print(f"total={len(df)}  benign={int((df.label==0).sum())}  malicious={int((df.label==1).sum())}")
print(f"train={len(tr)} val={len(va)} test={len(te)}")
print("DATASET TABLE ->", dict(benign=int((df.label==0).sum()),
      malicious=int((df.label==1).sum()), total=len(df), pct=round(100*df.label.mean(),1)))

## 5. Fine-tune BERT (change MODEL to compare variants)

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

MODEL = "bert-base-uncased"     # or "distilbert-base-uncased" or "ehsanaghaei/SecureBERT"
SAMPLE = 0                      # >0 = quick smoke run on a subset; 0 = full data
MAXLEN = 256

trd, vad = tr, va
if SAMPLE:
    trd = tr.groupby('label').head(SAMPLE//2)
    vad = va.groupby('label').head(SAMPLE//4)

tok = AutoTokenizer.from_pretrained(MODEL)
def mk(d):
    ds = Dataset.from_pandas(d[['text','label']], preserve_index=False)
    return ds.map(lambda b: tok(b['text'], truncation=True, max_length=MAXLEN,
                                padding='max_length'), batched=True)
train_ds, val_ds = mk(trd), mk(vad)

def metrics(p):
    pred=np.argmax(p.predictions,1); y=p.label_ids
    return dict(accuracy=accuracy_score(y,pred), precision=precision_score(y,pred,zero_division=0),
                recall=recall_score(y,pred,zero_division=0), f1=f1_score(y,pred,zero_division=0))

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments(output_dir="ckpt", num_train_epochs=3,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="f1", logging_steps=100, seed=SEED, report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  compute_metrics=metrics, callbacks=[EarlyStoppingCallback(2)])
trainer.train()
print("VALIDATION:", trainer.evaluate())

## 6. Evaluate: model / rules / hybrid

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score

def model_scores(texts, batch=64):
    model.eval(); dev=model.device; out=[]
    with torch.no_grad():
        for i in range(0,len(texts),batch):
            enc=tok(texts[i:i+batch],truncation=True,max_length=MAXLEN,padding=True,
                    return_tensors="pt").to(dev)
            pr=torch.softmax(model(**enc).logits,1)[:,1].cpu().numpy()
            out.extend(pr.tolist())
    return np.array(out)

def block(y,pred,score=None):
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    d=dict(P=round(precision_score(y,pred,zero_division=0),4),
           R=round(recall_score(y,pred,zero_division=0),4),
           F1=round(f1_score(y,pred,zero_division=0),4),
           FPR=round(fp/(fp+tn),4) if (fp+tn) else 0.0)
    if score is not None and len(set(y))>1: d['AUC']=round(roc_auc_score(y,score),4)
    return d

yte=te.label.to_numpy()
scores=model_scores(te.text.tolist())
pred_model=(scores>=0.5).astype(int)
pred_rules=np.array([rules_verdict(m,u) for m,u in zip(te.method,te.raw_uri)])
pred_hybrid=np.array([fuse(m,u,p) for m,u,p in zip(te.method,te.raw_uri,scores)])

R_model=block(yte,pred_model,scores); R_rules=block(yte,pred_rules); R_hybrid=block(yte,pred_hybrid,scores)
print("MODEL ",R_model); print("RULES ",R_rules); print("HYBRID",R_hybrid)

te2=te.copy(); te2['pred']=pred_hybrid
per_class={a:round(te2[(te2.label==1)&(te2.attack_type==a)].pred.mean(),4)
           for a in sorted(te2[te2.label==1].attack_type.unique())}
print("PER-CLASS RECALL:",per_class)

## 7. Classical baselines (TF-IDF + SVM / RF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
vec=TfidfVectorizer(analyzer="char_wb",ngram_range=(1,4),min_df=2,max_features=50000)
Xtr=vec.fit_transform(tr.text); Xte=vec.transform(te.text); ytr=tr.label.to_numpy()
svm=LinearSVC(random_state=SEED).fit(Xtr,ytr)
rf=RandomForestClassifier(n_estimators=200,n_jobs=-1,random_state=SEED).fit(Xtr,ytr)
R_svm=block(yte,svm.predict(Xte),svm.decision_function(Xte))
R_rf =block(yte,rf.predict(Xte),rf.predict_proba(Xte)[:,1])
print("TF-IDF+SVM",R_svm); print("TF-IDF+RF ",R_rf)

## 8. Threshold sensitivity (FPR vs recall)

In [ ]:
rows=[]
for t in np.round(np.arange(0.5,1.0,0.02),2):
    pr=(scores>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(yte,pr,labels=[0,1]).ravel()
    rows.append(dict(threshold=float(t),
        recall=round(recall_score(yte,pr,zero_division=0),4),
        fpr=round(fp/(fp+tn),4) if (fp+tn) else 0.0,
        f1=round(f1_score(yte,pr,zero_division=0),4)))
thr=pd.DataFrame(rows); print(thr.to_string(index=False))
thr.to_csv("threshold.csv",index=False)
best=thr.loc[thr.f1.idxmax()]; print("\nBest-F1 point:",dict(best))

## 9. Latency (per-request model inference)

In [ ]:
import time
sample=te.text.tolist()[:1000]
_=model_scores(sample[:16])
lat=[]; model.eval(); dev=model.device
with torch.no_grad():
    for txt in sample:
        s=time.perf_counter()
        enc=tok([txt],truncation=True,max_length=MAXLEN,padding=True,return_tensors="pt").to(dev)
        _=model(**enc).logits
        if dev.type=="cuda": torch.cuda.synchronize()
        lat.append((time.perf_counter()-s)*1000)
lat=np.array(lat)
LAT=dict(device=dev.type, p50=round(np.percentile(lat,50),2),
         p95=round(np.percentile(lat,95),2), p99=round(np.percentile(lat,99),2),
         mean=round(lat.mean(),2), rps=round(1000/lat.mean(),1))
print("LATENCY:",LAT)

## 10. Adversarial evasion (semantics-preserving mutations)

In [ ]:
from urllib.parse import quote
def m_comment(u): return re.sub(r"\s+","/**/",u)
def m_case(u): return "".join(c.upper() if random.random()>.5 else c.lower() for c in u)
def m_url(u): return quote(u,safe="/?=&%")
def m_dbl(u): return quote(quote(u,safe=""),safe="")
def m_ws(u): return u.replace(" ","%09")
def m_junk(u):
    sep="&" if "?" in u else "?"; return f"{u}{sep}x={random.randint(1000,9999)}"
MUT={"inline_comment":m_comment,"case":m_case,"url_encode":m_url,
     "double_encode":m_dbl,"whitespace":m_ws,"junk_param":m_junk}

mal=te[te.label==1]
if len(mal)>300: mal=mal.sample(300,random_state=SEED)
base=[(m,u) for m,u,p in zip(mal.method,mal.raw_uri,
      model_scores([preprocess_request(m,u) for m,u in zip(mal.method,mal.raw_uri)])) if fuse(m,u,p)]
n=len(base); ADV={}
for name,fn in MUT.items():
    mut=[fn(u) for _,u in base]
    ps=model_scores([preprocess_request(m,u) for (m,_),u in zip(base,mut)])
    ev=sum(1 for (m,_),u,p in zip(base,mut,ps) if not fuse(m,u,p))
    ADV[name]=round(ev/n,4) if n else 0.0
ADV["OVERALL"]=round(np.mean(list(ADV.values())),4)
print("initially blocked:",n); print("EVASION:",ADV)

## 11. Results summary - copy into the paper

In [ ]:
def row(name,d): return f"{name:22s} P={d.get('P','-')} R={d.get('R','-')} F1={d.get('F1','-')} FPR={d.get('FPR','-')} AUC={d.get('AUC','-')}"
print("="*68); print("MAIN + ABLATION (Tables III/IV)"); print("="*68)
print(row("Rules only",R_rules)); print(row("TF-IDF+SVM",R_svm)); print(row("TF-IDF+RF",R_rf))
print(row("BERT model-only",R_model)); print(row("HYBRID (ours)",R_hybrid))
print("\nPER-CLASS RECALL:",per_class)
print("LATENCY (Table VI):",LAT)
print("ADVERSARIAL (Table VII):",ADV)
print("\nthreshold.csv saved (download for the threshold figure).")
print(">>> Replace matching red [PLACEHOLDER] cells in paper/journal/main.tex")

### Where each number goes
| Printed here | Paper location |
|---|---|
| DATASET TABLE | Table II |
| MAIN + ABLATION rows | Tables III & IV |
| PER-CLASS RECALL | per-class chart |
| LATENCY | Table VI |
| ADVERSARIAL | Table VII |
| threshold.csv | threshold figure |

Re-run Section 5 with `MODEL="distilbert-base-uncased"` or
`"ehsanaghaei/SecureBERT"` to fill those rows. For cross-dataset (Table V),
upload a second dataset in Section 2 and re-run Sections 4 & 6.